E-commerce Text Classification with RNNs and Multiple Embedding Techniques

In [ ]:
!pip install gensim

In [3]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


In [5]:
import csv

# Use quoting=csv.QUOTE_NONE to handle unclosed quotes (like inch marks) in descriptions
# The dataset typically doesn't have a header, so we provide names
train_df = pd.read_csv(
    "/content/drive/MyDrive/ecommerce/ecommerceDataset.csv",
    names=["label", "text"],
    header=None,
    quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

print(f"Loaded {len(train_df)} rows.")
train_df.head()

Loaded 14204 rows.


label  \
Household "Paper Plane Design Framed Wall Hanging Motivat... you would be proud to possess this unique paint... with only the and inks and precision epson          roland and hp printers. This innovative hd pri...  to achieve brilliant and true colours. Due to ...   our prints retain their beautiful colours for...   
          "SAF 'Floral' Framed Painting (Wood                30 inch x 10 inch                                  Special Effect UV Print Textured                    SAO297) Painting made up in synthetic frame wi... NaN                                                                                               NaN   
          "SAF 'UV Textured Modern Art Print Framed' Pain... 35 cm x 50 cm x 3 cm                               Set of 3) Color:Multicolor                     ...  the end product will be a picture that can spe...  it does not include glass along with the frame...                and shopping is just a click away!"   
          "Incredible Gifts India Wooden Happy Birthday U... Which Is Quite Solid With Light Particle Patter... Some Can Be Used As Table Top And The Big Sizes... NaN                                                NaN                                                                                               NaN   
          "Paper Plane Design Starry Night Vangoh Wall Ar... with only the best and original inks and precis... to achieve brilliant and true colours. Due to t...  our Canvas prints retain their beautiful colou...  to ensure that the colours of your original im...   with brilliant tones. Add colour and style to...   

                                                                                                                                                                                                                                                                                                                       text  
Household "Paper Plane Design Framed Wall Hanging Motivat... you would be proud to possess this unique paint... with only the and inks and precision epson          roland and hp printers. This innovative hd pri...  to achieve brilliant and true colours. Due to ...                               office or any room."  
          "SAF 'Floral' Framed Painting (Wood                30 inch x 10 inch                                  Special Effect UV Print Textured                    SAO297) Painting made up in synthetic frame wi... NaN                                                                                               NaN  
          "SAF 'UV Textured Modern Art Print Framed' Pain... 35 cm x 50 cm x 3 cm                               Set of 3) Color:Multicolor                     ...  the end product will be a picture that can spe...  it does not include glass along with the frame...                                                NaN  
          "Incredible Gifts India Wooden Happy Birthday U... Which Is Quite Solid With Light Particle Patter... Some Can Be Used As Table Top And The Big Sizes... NaN                                                NaN                                                                                               NaN  
          "Paper Plane Design Starry Night Vangoh Wall Ar... with only the best and original inks and precis... to achieve brilliant and true colours. Due to t...  our Canvas prints retain their beautiful colou...  to ensure that the colours of your original im...   office or any room. A perfect size of 36 inch...

In [6]:
print(train_df.shape)
print(train_df['label'].value_counts())


(14204, 2)
label
ready for use."                                                                                                                                                                                                    14
India"                                                                                                                                                                                                             11
exchanges or any other issue                                                                                                                                                                                       11
designing and improving our products to ensure quality and maximum function. Maintaining our standing of being the world's best at what we do is important to us and what you expect from a name like STANLEY."     9
it easily supports up to 150 kgs."                                                                                             

In [8]:
def clean_text(text):
    # Check if text is a string to avoid AttributeErrors with NaN/float values
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

# Drop rows with missing values and use .copy() to avoid SettingWithCopyWarning
train_df = train_df.dropna(subset=["text"]).copy()
train_df["clean_text"] = train_df["text"].apply(clean_text)

/tmp/ipython-input-3150569572.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["clean_text"] = train_df["text"].apply(clean_text)


In [9]:
le = LabelEncoder()
y = le.fit_transform(train_df["label"])
y = to_categorical(y)

num_classes = y.shape[1]


EXPERIMENT 1: RNN + TF-IDF

In [10]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(train_df["clean_text"]).toarray()


In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


In [12]:
X_train_rnn = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val_rnn   = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))

model_tfidf = Sequential([
    SimpleRNN(128, input_shape=(1, X_train.shape[1])),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model_tfidf.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=1e-3),
    metrics=["accuracy"]
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [19]:
history_tfidf = model_tfidf.fit(
    X_train_rnn, y_train,
    validation_data=(X_val_rnn, y_val),
    epochs=50,
    batch_size=64
)


Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9775 - loss: 0.0812 - val_accuracy: 0.7048 - val_loss: 2.1852
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9879 - loss: 0.0677 - val_accuracy: 0.7048 - val_loss: 2.1889
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9823 - loss: 0.0663 - val_accuracy: 0.7048 - val_loss: 2.1921
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9798 - loss: 0.0678 - val_accuracy: 0.7048 - val_loss: 2.1938
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9874 - loss: 0.0663 - val_accuracy: 0.7048 - val_loss: 2.1946
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9875 - loss: 0.0673 - val_accuracy: 0.7048 - val_loss: 2.1963
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9830 - loss: 0.0679 - val_accuracy: 0.7048 - val_loss: 2.1976
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9816 - loss: 0.0596 - val_accuracy: 0.7048 - v

In [16]:
tfidf_results = {
    "Embedding": "TF-IDF",
    "RNN units": 128,
    "Dropout": 0.5,
    "Learning rate": 1e-3,
    "Best Val Accuracy": max(history_tfidf.history["val_accuracy"])
}

tfidf_results


{'Embedding': 'TF-IDF',
 'RNN units': 128,
 'Dropout': 0.5,
 'Learning rate': 0.001,
 'Best Val Accuracy': 0.699999988079071}

EXPERIMENT 2: RNN + Skip-gram (Word2Vec)

In [17]:
sentences = [text.split() for text in train_df["clean_text"]]


In [20]:
w2v_skipgram = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,   # Skip-gram
    epochs=50
)


In [21]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["clean_text"])

sequences = tokenizer.texts_to_sequences(train_df["clean_text"])
max_len = 50
X_seq = pad_sequences(sequences, maxlen=max_len)


In [22]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():
    if word in w2v_skipgram.wv:
        embedding_matrix[idx] = w2v_skipgram.wv[word]


In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y, test_size=0.2, random_state=42
)


In [24]:
model_skipgram = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),
    LSTM(128),
    Dropout(0.4),
    Dense(num_classes, activation="softmax")
])

model_skipgram.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(1e-3),
    metrics=["accuracy"]
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [27]:
history_skipgram = model_skipgram.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25,
    batch_size=64
)


Epoch 1/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 240ms/step - accuracy: 0.9435 - loss: 0.2391 - val_accuracy: 0.6667 - val_loss: 3.2726
Epoch 2/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 256ms/step - accuracy: 0.9564 - loss: 0.2423 - val_accuracy: 0.6667 - val_loss: 3.2413
Epoch 3/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step - accuracy: 0.9440 - loss: 0.2301 - val_accuracy: 0.6667 - val_loss: 3.1987
Epoch 4/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 194ms/step - accuracy: 0.9376 - loss: 0.2607 - val_accuracy: 0.6667 - val_loss: 3.2554
Epoch 5/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 203ms/step - accuracy: 0.9353 - loss: 0.2478 - val_accuracy: 0.6619 - val_loss: 3.2835
Epoch 6/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 202ms/step - accuracy: 0.9373 - loss: 0.2520 - val_accuracy: 0.6714 - val_loss: 3.3076
Epoch 7/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 5s 186ms/step - accuracy: 0.9438 - loss: 0.2406 - val_accuracy: 0.6714 - val_loss: 3.2887
Epoch 8/25
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.9319 - loss: 0.2552 - val_accuracy: 0.6

In [28]:
skipgram_results = {
    "Embedding": "Word2Vec Skip-gram",
    "Embedding dim": 100,
    "RNN units": 128,
    "Dropout": 0.4,
    "Best Val Accuracy": max(history_skipgram.history["val_accuracy"])
}

skipgram_results


{'Embedding': 'Word2Vec Skip-gram',
 'Embedding dim': 100,
 'RNN units': 128,
 'Dropout': 0.4,
 'Best Val Accuracy': 0.6761904954910278}

EXPERIMENT 3: RNN + CBOW (Word2Vec)

In [29]:
w2v_cbow = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0,   # CBOW
    epochs=10
)


In [30]:
embedding_matrix_cbow = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():
    if word in w2v_cbow.wv:
        embedding_matrix_cbow[idx] = w2v_cbow.wv[word]


In [31]:
model_cbow = Sequential([
    Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix_cbow],
        input_length=max_len,
        trainable=False
    ),
    LSTM(128),
    Dropout(0.4),
    Dense(num_classes, activation="softmax")
])

model_cbow.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(1e-3),
    metrics=["accuracy"]
)


In [33]:
history_cbow = model_cbow.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64
)


Epoch 1/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.0754 - loss: 4.8205 - val_accuracy: 0.0571 - val_loss: 5.9932
Epoch 2/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 97ms/step - accuracy: 0.0778 - loss: 4.7488 - val_accuracy: 0.0619 - val_loss: 5.9192
Epoch 3/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - accuracy: 0.0821 - loss: 4.6386 - val_accuracy: 0.0571 - val_loss: 5.8568
Epoch 4/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 99ms/step - accuracy: 0.0860 - loss: 4.5417 - val_accuracy: 0.0619 - val_loss: 5.8598
Epoch 5/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 2s 160ms/step - accuracy: 0.0997 - loss: 4.5007 - val_accuracy: 0.0857 - val_loss: 5.7903
Epoch 6/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 178ms/step - accuracy: 0.1020 - loss: 4.4094 - val_accuracy: 0.0857 - val_loss: 5.7649
Epoch 7/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step - accuracy: 0.1120 - loss: 4.3269 - val_accuracy: 0.0857 - val_loss: 5.7294
Epoch 8/100
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - accuracy: 0.1357 - loss: 4.1835 - val_accurac

In [34]:
cbow_results = {
    "Embedding": "Word2Vec CBOW",
    "Embedding dim": 100,
    "RNN units": 128,
    "Dropout": 0.4,
    "Best Val Accuracy": max(history_cbow.history["val_accuracy"])
}

cbow_results


{'Embedding': 'Word2Vec CBOW',
 'Embedding dim': 100,
 'RNN units': 128,
 'Dropout': 0.4,
 'Best Val Accuracy': 0.5904762148857117}

Comparison table

In [35]:
pd.DataFrame([tfidf_results, skipgram_results, cbow_results])


,Embedding,RNN units,Dropout,Learning rate,Best Val Accuracy,Embedding dim
0,TF-IDF,128,0.5,0.001,0.700000,NaN
1,Word2Vec Skip-gram,128,0.4,NaN,0.676190,100.0
2,Word2Vec CBOW,128,0.4,NaN,0.590476,100.0
